# baseline — frozen-backbone ablation of exp04 (no-fine-tuning baseline)

**Purpose (one variable under test):** measure how much of exp04's part-segmentation
performance comes from fine-tuning the backbone, by training the *same* `yolov8s-seg.pt`
starting point on the *same* exp04 HITL split and recipe, with the backbone frozen so
only the neck + segmentation head trains. This is the "no-fine-tuning baseline" ablation
for the paper: everything is held identical to exp04 except one variable — whether the
backbone's weights are allowed to update.

**Everything reused from exp04, nothing regenerated:**
- Same pretrained start: `yolov8s-seg.pt`.
- Same dataset/split: reads exp04's own `data.yaml` and physically materialized
  `train/val/test` folders on Drive (798/99/101 images) — this notebook never creates a
  new split, never re-downloads HITL, never re-converts annotations.
- Same hyperparameters and augmentation: read programmatically from exp04's saved
  `config.yaml` (Section 2), not retyped, so this can't silently desync from exp04's
  recipe.
- Same evaluation recipe and the same panel-accuracy formula as exp04 Section 8a.

**The one and only change:** `freeze=10` — freezes the entire yolov8s-seg backbone
(layers 0–9: the Conv/C2f/SPPF stem) so only the neck + segmentation head (layer index
≥10) update. Section 3 inspects the *live* loaded model's layer list and asserts that
layer 10 is the first non-backbone layer before training starts — the freeze boundary is
checked against the actual checkpoint, not just asserted in a comment. A training
callback (Section 4) additionally forces BatchNorm layers inside the frozen range into
`eval()` every epoch: Ultralytics' `freeze=N` stops gradient flow into those weights but
does **not** by itself stop their BatchNorm running statistics from drifting during
forward passes — without the callback this would not be a fully frozen backbone.

**Safety:** this notebook only *reads* `exp04_hitl_panels/config.yaml`,
`split_info.yaml`, and `data.yaml`, and only *reads* `exp04_hitl_panels/weights/best.pt`
(once, read-only, in Section 7, to recompute directly-comparable "final" numbers under
this same session). It never writes into `experiments/exp04_hitl_panels/` or
`datasets/<version>/`. All baseline outputs go to a fresh `experiments/baseline/`
directory.

**Good practices enforced (for the paper):**
- **Deterministic:** `seed` is read from exp04's config (Section 2) and passed unchanged
  to `model.train()` (Section 4).
- **Fair comparison:** data, split, and every hyperparameter are read from exp04's own
  saved files (Section 2), never retyped; the only intentional difference is `freeze=10`
  (Section 3/4).
- **No test leakage:** the 101-image test split is evaluated exactly once per checkpoint,
  after training, under the final weights (Section 6) — and, separately, once read-only
  against exp04's already-finalized `best.pt` (Section 7), which is not a new
  training/tuning event.
- **Reproducibility:** `config.yaml` (Section 9) records every hyperparameter, the
  freeze setting, the frozen layer/BatchNorm names, and library versions (`ultralytics`,
  `torch`, Python, GPU).
- **Box and mask metrics both reported, never just one** (Sections 6/7/8).
- **exp04 is never modified** — only its `config.yaml`, `split_info.yaml`, `data.yaml`,
  and `weights/best.pt` are read.

## 📁 Section 1 — Setup: mount Drive, install deps
Independently re-runnable, same pattern as exp04.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HUB_DISABLE_XET'] = '1'  # avoids a flaky xet handshake seen in earlier experiments; harmless if unused here

!pip install -q ultralytics

import ultralytics
ultralytics.checks()

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 42.4/112.6 GB disk)


## 🔗 Section 2 — Load exp04's config, split, and data.yaml (read-only — nothing regenerated)

Reads exp04's saved `config.yaml` and `split_info.yaml` for every hyperparameter,
augmentation setting, and the dataset location, and points straight at exp04's
already-materialized `data.yaml`. If any of these files are missing, this notebook stops
rather than silently falling back to re-deriving values — the whole point of this
baseline is that nothing about the data or recipe is retyped or regenerated.

In [4]:
import os, yaml, json

DRIVE_ROOT = '/content/drive/MyDrive/carparts/segmentation'
EXP04_ID   = 'exp04_hitl_panels'
EXP04_DIR  = os.path.join(DRIVE_ROOT, 'experiments', EXP04_ID)

BASELINE_ID  = 'baseline'
BASELINE_DIR = os.path.join(DRIVE_ROOT, 'experiments', BASELINE_ID)
for sub in ['weights', 'confusion_matrix', 'final_recompute_eval']:
    os.makedirs(os.path.join(BASELINE_DIR, sub), exist_ok=True)

exp04_config_path = os.path.join(EXP04_DIR, 'config.yaml')
exp04_split_info_path = os.path.join(EXP04_DIR, 'split_info.yaml')
exp04_best_pt_path = os.path.join(EXP04_DIR, 'weights', 'part_seg_exp04.pt')

for required_path, what in [
    (exp04_config_path, 'exp04 config.yaml'),
    (exp04_split_info_path, 'exp04 split_info.yaml'),
    (exp04_best_pt_path, 'part_seg_exp04.pt'),
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            f'Missing {what} at {required_path}. This baseline reuses exp04\'s recipe and '
            'split exactly and refuses to regenerate them -- run exp04_hitl_panels.ipynb '
            'to completion first.'
        )

with open(exp04_config_path) as f:
    EXP04_CONFIG = yaml.safe_load(f)
with open(exp04_split_info_path) as f:
    EXP04_SPLIT_INFO = yaml.safe_load(f)

# ---- every hyperparameter below comes from exp04's own saved config, not retyped ----
MODEL_SOURCE      = EXP04_CONFIG['model_source']
IMG_SIZE          = EXP04_CONFIG['imgsz']
EPOCHS            = EXP04_CONFIG['epochs']
PATIENCE          = EXP04_CONFIG['patience']
WEIGHT_DECAY      = EXP04_CONFIG['weight_decay']
BATCH             = EXP04_CONFIG['batch']
SEED              = EXP04_CONFIG['seed']
AUGMENT           = EXP04_CONFIG['augment']
CANONICAL_CLASSES = EXP04_CONFIG['canonical_classes']
PANEL_CLASSES     = set(EXP04_CONFIG['panel_classes'])
CONF_THRESH       = EXP04_CONFIG['conf_thresh']
IOU_THRESH        = EXP04_CONFIG['iou_thresh']
DATASET_VERSION   = EXP04_CONFIG['dataset_version']

assert MODEL_SOURCE == 'yolov8s-seg.pt', (
    f'exp04\'s config.yaml records model_source={MODEL_SOURCE!r}, not \'yolov8s-seg.pt\' -- '
    'the baseline brief assumes exp04 started from the generic pretrained seg checkpoint; '
    'stopping rather than silently training from a different start point.'
)
assert len(CANONICAL_CLASSES) == 16 and len(PANEL_CLASSES) == 8

DATASET_DIR = os.path.join(DRIVE_ROOT, 'datasets', DATASET_VERSION)
DATA_YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.exists(DATA_YAML_PATH):
    raise FileNotFoundError(
        f'exp04\'s data.yaml not found at {DATA_YAML_PATH} -- the physically materialized '
        'split exp04 built must exist on Drive before running this baseline.'
    )

print('Loaded exp04 config from:', exp04_config_path)
print(f'  model_source={MODEL_SOURCE}  imgsz={IMG_SIZE}  epochs={EPOCHS}  patience={PATIENCE}')
print(f'  weight_decay={WEIGHT_DECAY}  batch={BATCH}  seed={SEED}')
print(f'  augment={AUGMENT}')
print(f'  conf_thresh={CONF_THRESH}  iou_thresh={IOU_THRESH}')
print('DATA_YAML_PATH (reused, not regenerated):', DATA_YAML_PATH)

Loaded exp04 config from: /content/drive/MyDrive/carparts/segmentation/experiments/exp04_hitl_panels/config.yaml
  model_source=yolov8s-seg.pt  imgsz=640  epochs=150  patience=20
  weight_decay=0.0005  batch=16  seed=42
  augment={'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.2, 'scale': 0.8, 'shear': 0.0, 'perspective': 0.0, 'fliplr': 0.5, 'flipud': 0.0, 'mosaic': 1.0, 'close_mosaic': 10, 'mixup': 0.0, 'copy_paste': 0.0}
  conf_thresh=0.25  iou_thresh=0.7
DATA_YAML_PATH (reused, not regenerated): /content/drive/MyDrive/carparts/segmentation/datasets/v3_hitl/data.yaml


In [5]:
from pathlib import Path

EXPECTED_COUNTS = EXP04_SPLIT_INFO['counts']  # exp04's own recorded split sizes
actual_counts = {}
for split in ('train', 'val', 'test'):
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"exp04's materialized '{split}' split missing at {img_dir}")
    actual_counts[split] = sum(1 for p in Path(img_dir).glob('*') if p.is_file())

print("Split sizes (from exp04's materialized folders on Drive):")
for split in ('train', 'val', 'test'):
    print(f'  {split}: {actual_counts[split]} images (exp04 recorded {EXPECTED_COUNTS[split]})')

if actual_counts != EXPECTED_COUNTS:
    raise RuntimeError(
        f"Split counts on Drive {actual_counts} do not match exp04's own split_info.yaml "
        f'{EXPECTED_COUNTS} -- the dataset on Drive has changed since exp04 ran. Investigate '
        'before training; this baseline must not train on a different split than exp04.'
    )

TOTAL = sum(actual_counts.values())
print(f'\nTotal {TOTAL} images -- train/val/test = '
      f'{actual_counts["train"]}/{actual_counts["val"]}/{actual_counts["test"]}')
if (actual_counts['train'], actual_counts['val'], actual_counts['test']) != (798, 99, 101):
    print("Note: counts differ from the 798/99/101 cited in the baseline brief -- still "
          "proceeding since they match exp04's own recorded split_info.yaml exactly, which "
          "is the authority this notebook defers to, but flagging the mismatch.")
else:
    print('Matches the 798/99/101 split specified in the baseline brief.')

Split sizes (from exp04's materialized folders on Drive):
  train: 798 images (exp04 recorded 798)
  val: 99 images (exp04 recorded 99)
  test: 101 images (exp04 recorded 101)

Total 998 images -- train/val/test = 798/99/101
Matches the 798/99/101 split specified in the baseline brief.


## 🧊 Section 3 — Confirm the backbone/head boundary before freezing

`freeze=10` is the standard "freeze the whole backbone" recipe for YOLOv8(-seg), but
it's confirmed here against the *actual* loaded `yolov8s-seg.pt` layer list rather than
assumed — if a future Ultralytics/model version ever changes the layer ordering, this
cell fails loudly instead of silently freezing the wrong layers.

In [6]:
from ultralytics import YOLO
import torch

FREEZE_LAYERS = 10  # first 10 top-level layers (indices 0-9) = the yolov8s-seg backbone

_probe = YOLO(MODEL_SOURCE)
_seq = _probe.model.model  # nn.Sequential of top-level layers, indices 0..N

print(f'{MODEL_SOURCE} top-level layer list ({len(_seq)} layers):')
for i, layer in enumerate(_seq):
    tag = '  <-- first layer NOT frozen (neck/head starts here)' if i == FREEZE_LAYERS else ''
    print(f'  [{i:2d}] {layer.__class__.__name__}{tag}')

# The yolov8-seg backbone (Conv/C2f stem + SPPF) is layers 0-9; the neck starts with the
# first upsample/concat layer. Assert that boundary actually holds for this checkpoint
# before relying on freeze=10 to mean "exactly the backbone".
_backbone_layer_types = {_seq[i].__class__.__name__ for i in range(FREEZE_LAYERS)}
_first_head_layer_type = _seq[FREEZE_LAYERS].__class__.__name__
assert 'SPPF' in _backbone_layer_types, (
    f'Expected an SPPF layer within the first {FREEZE_LAYERS} layers (end of the YOLOv8-seg '
    f'backbone) -- found layer types {sorted(_backbone_layer_types)}. The freeze={FREEZE_LAYERS} '
    'boundary may not match this checkpoint\'s architecture; inspect the printout above before training.'
)
assert _first_head_layer_type in ('Upsample', 'Concat'), (
    f'Expected layer [{FREEZE_LAYERS}] to be the first neck layer (Upsample/Concat) -- got '
    f'{_first_head_layer_type!r} instead. freeze={FREEZE_LAYERS} would not line up with the '
    'backbone/head boundary for this checkpoint; inspect the printout above before training.'
)
print(f'\nConfirmed: layers [0..{FREEZE_LAYERS-1}] are the backbone (ends in SPPF), '
      f'layer [{FREEZE_LAYERS}] ({_first_head_layer_type}) is the first neck/head layer.')

FROZEN_MODULE_NAMES = [
    name for name, _ in _probe.model.named_modules()
    if name and any(name == f'model.{i}' or name.startswith(f'model.{i}.') for i in range(FREEZE_LAYERS))
]
FROZEN_BN_MODULE_NAMES = [
    name for name, m in _probe.model.named_modules()
    if name in FROZEN_MODULE_NAMES and isinstance(m, torch.nn.BatchNorm2d)
]
print(f'\n{len(FROZEN_MODULE_NAMES)} submodules fall under the frozen prefixes '
      f'(model.0. .. model.{FREEZE_LAYERS-1}.), including {len(FROZEN_BN_MODULE_NAMES)} '
      'BatchNorm2d layers that the training callback below will keep in eval() mode.')

del _probe  # only used to inspect the architecture; training loads its own fresh copy

yolov8s-seg.pt top-level layer list (23 layers):
  [ 0] Conv
  [ 1] Conv
  [ 2] C2f
  [ 3] Conv
  [ 4] C2f
  [ 5] Conv
  [ 6] C2f
  [ 7] Conv
  [ 8] C2f
  [ 9] SPPF
  [10] Upsample  <-- first layer NOT frozen (neck/head starts here)
  [11] Concat
  [12] C2f
  [13] Upsample
  [14] Concat
  [15] C2f
  [16] Conv
  [17] Concat
  [18] C2f
  [19] Conv
  [20] Concat
  [21] C2f
  [22] Segment

Confirmed: layers [0..9] are the backbone (ends in SPPF), layer [10] (Upsample) is the first neck/head layer.

98 submodules fall under the frozen prefixes (model.0. .. model.9.), including 27 BatchNorm2d layers that the training callback below will keep in eval() mode.


## 🏋️ Section 4 — Train with the backbone frozen (single run)

Identical call to exp04's training cell — same `data.yaml`, same hyperparameters and
augmentation (all loaded from exp04's config in Section 2) — with exactly one addition:
`freeze=FREEZE_LAYERS`. A callback keeps the frozen backbone's BatchNorm layers in
`eval()` every epoch (see Section 0/3) so this is genuinely a frozen-backbone run, not
just frozen weights with still-drifting batch statistics. Resume-aware like exp04: safe
to re-run after a disconnect.

In [7]:
RUNS_PROJECT = os.path.join(BASELINE_DIR, 'runs')
RUN_NAME = BASELINE_ID
last_pt = os.path.join(RUNS_PROJECT, RUN_NAME, 'weights', 'last.pt')

def _freeze_backbone_bn(trainer):
    """Keep frozen-backbone BatchNorm layers in eval() every epoch.

    trainer.model.train() (called at the start of every epoch by Ultralytics) puts the
    WHOLE model back into train mode, including frozen BN layers -- freeze=N alone only
    stops their gradients, not their running-stat updates. Re-forcing eval() here each
    epoch is what makes this an actually-frozen backbone.
    """
    n = 0
    for name, module in trainer.model.named_modules():
        if name in FROZEN_BN_MODULE_NAMES:
            module.eval()
            n += 1
    if trainer.epoch == 0:
        print(f'[freeze-bn callback] {n} frozen BatchNorm2d layers forced to eval()')

if os.path.exists(last_pt):
    print(f'Found existing checkpoint at {last_pt} -- resuming training.')
    model = YOLO(last_pt)
    model.add_callback('on_train_epoch_start', _freeze_backbone_bn)
    try:
        train_results = model.train(resume=True)
    except Exception as e:
        if 'already' in str(e).lower():
            print('Training was already complete:', e)
            train_results = None
        else:
            raise
else:
    model = YOLO(MODEL_SOURCE)
    model.add_callback('on_train_epoch_start', _freeze_backbone_bn)
    train_results = model.train(
        data=DATA_YAML_PATH,          # exp04's data.yaml, reused as-is
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        seed=SEED,
        weight_decay=WEIGHT_DECAY,
        batch=BATCH,
        freeze=FREEZE_LAYERS,          # <- the one and only recipe change vs exp04
        project=RUNS_PROJECT,
        name=RUN_NAME,
        exist_ok=True,
        save_period=1,                 # last.pt lands on Drive every epoch -- survives a disconnect mid-run
        **AUGMENT,
    )

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/carparts/segmentation/datasets/v3_hitl/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bas

In [8]:
import shutil

run_dir = os.path.join(RUNS_PROJECT, RUN_NAME)
for fname in ('part_seg_exp04.pt', 'last.pt'):
    src = os.path.join(run_dir, 'weights', fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BASELINE_DIR, 'weights', fname))

for fname in ('results.csv', 'results.png'):
    src = os.path.join(run_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BASELINE_DIR, fname))

BASELINE_BEST_PT = os.path.join(BASELINE_DIR, 'weights', 'part_seg_exp04.pt')
print('BASELINE_BEST_PT:', BASELINE_BEST_PT,
      '(exists)' if os.path.exists(BASELINE_BEST_PT) else '(MISSING -- check the training cell above)')

BASELINE_BEST_PT: /content/drive/MyDrive/carparts/segmentation/experiments/baseline/weights/best.pt (exists)


## 📊 Section 5 — Shared evaluation function (box + mask, per-class, confusion matrix, panel accuracy)

One function, used twice: once on the baseline's own frozen-backbone checkpoint
(Section 6), and once on exp04's own `best.pt` loaded read-only (Section 7), so
"baseline" and "final" numbers in `baseline_vs_final.csv` come from the exact same
evaluation code, run in the same session, on the exact same 101-image test split — not
two numbers computed months apart under potentially different library versions.

Panel accuracy uses the **same formula as exp04 Section 8a**, stated explicitly here and
again in every saved summary:

> `panel_accuracy = 100 * (sum of confusion-matrix diagonal entries for the 8 panel
> classes) / (sum of all TRUE instances of panel classes, i.e. each panel class's
> confusion-matrix column total including the 'background' row = missed detections)`

In [9]:
import pandas as pd
import numpy as np

def evaluate_checkpoint(weights_path, tag, out_dir):
    """Runs the official Ultralytics test-split val() once, then derives every metric
    this evaluation reports from its output -- box + mask overall, per-class AP50 and
    AP50-95 for all 16 classes, raw + normalized confusion matrix, and panel accuracy
    (exp04's exact formula). Writes everything under out_dir. Read-only on weights_path."""
    os.makedirs(out_dir, exist_ok=True)
    eval_model = YOLO(weights_path)
    test_metrics = eval_model.val(
        data=DATA_YAML_PATH,
        split='test',
        imgsz=IMG_SIZE,
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        project=out_dir,
        name='test_eval',
        plots=True,
        exist_ok=True,
    )

    box, seg = test_metrics.box, test_metrics.seg
    overall = {
        'box':  {'precision': float(box.mp), 'recall': float(box.mr),
                 'map50': float(box.map50), 'map50_95': float(box.map)},
        'mask': {'precision': float(seg.mp), 'recall': float(seg.mr),
                 'map50': float(seg.map50), 'map50_95': float(seg.map)},
    }

    # per-class AP50 / AP50-95, box and mask, for all 16 canonical classes
    per_class_rows = []
    box_idx = {int(c): i for i, c in enumerate(box.ap_class_index)}
    seg_idx = {int(c): i for i, c in enumerate(seg.ap_class_index)}
    for cls_id, cls_name in enumerate(CANONICAL_CLASSES):
        bi, si = box_idx.get(cls_id), seg_idx.get(cls_id)
        per_class_rows.append({
            'class': cls_name,
            'is_panel': cls_name in PANEL_CLASSES,
            'box_ap50':     float(box.ap50[bi]) if bi is not None else None,
            'box_ap50_95':  float(box.ap[bi])   if bi is not None else None,
            'mask_ap50':    float(seg.ap50[si]) if si is not None else None,
            'mask_ap50_95': float(seg.ap[si])   if si is not None else None,
        })
    per_class_df = pd.DataFrame(per_class_rows).sort_values('is_panel', ascending=False)
    per_class_csv = os.path.join(out_dir, f'{tag}_per_class_metrics.csv')
    per_class_df.to_csv(per_class_csv, index=False)

    # confusion matrix: raw + column-normalized (same normalization Ultralytics' own
    # plot(normalize=True) uses), full 16+background
    cm_raw = test_metrics.confusion_matrix.matrix
    names = CANONICAL_CLASSES + ['background']
    cm_raw_df = pd.DataFrame(cm_raw, index=names, columns=names)
    cm_raw_csv = os.path.join(out_dir, f'{tag}_confusion_matrix_raw.csv')
    cm_raw_df.to_csv(cm_raw_csv)

    cm_norm = cm_raw / (cm_raw.sum(axis=0, keepdims=True) + 1e-9)
    cm_norm_df = pd.DataFrame(cm_norm, index=names, columns=names)
    cm_norm_csv = os.path.join(out_dir, f'{tag}_confusion_matrix_normalized.csv')
    cm_norm_df.to_csv(cm_norm_csv)

    panel_names = [c for c in CANONICAL_CLASSES if c in PANEL_CLASSES]
    panel_tp = pd.Series({p: cm_raw_df.loc[p, p] for p in panel_names})
    true_panel_totals = (cm_raw_df.loc[panel_names, panel_names].sum(axis=0)
                          + cm_raw_df.loc['background', panel_names])
    panel_accuracy_pct = (100.0 * panel_tp.sum() / true_panel_totals.sum()
                           if true_panel_totals.sum() else float('nan'))

    # locate the confusion-matrix PNGs Ultralytics plots=True already saved
    run_save_dir = os.path.join(out_dir, 'test_eval')
    cm_png = os.path.join(run_save_dir, 'confusion_matrix.png')
    cm_norm_png = os.path.join(run_save_dir, 'confusion_matrix_normalized.png')

    summary = {
        'tag': tag,
        'weights_path': weights_path,
        'test_images': actual_counts['test'],
        'conf_thresh': CONF_THRESH,
        'iou_thresh': IOU_THRESH,
        'overall': overall,
        'per_class': per_class_rows,
        'panel_accuracy_pct': float(panel_accuracy_pct),
        'panel_accuracy_formula': (
            '100 * sum(confusion_matrix diagonal for the 8 panel classes) / '
            'sum(confusion_matrix column totals for the 8 panel classes, including the '
            "'background' row for missed detections) -- identical definition to exp04 "
            'Section 8a'
        ),
        'confusion_matrix_raw_csv': cm_raw_csv,
        'confusion_matrix_normalized_csv': cm_norm_csv,
        'confusion_matrix_raw_png': cm_png if os.path.exists(cm_png) else None,
        'confusion_matrix_normalized_png': cm_norm_png if os.path.exists(cm_norm_png) else None,
        'per_class_csv': per_class_csv,
    }
    summary_path = os.path.join(out_dir, f'{tag}_metrics_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    summary['summary_path'] = summary_path

    print(f'[{tag}] box mAP50/50-95: {overall["box"]["map50"]:.4f}/{overall["box"]["map50_95"]:.4f}  '
          f'mask mAP50/50-95: {overall["mask"]["map50"]:.4f}/{overall["mask"]["map50_95"]:.4f}  '
          f'panel_accuracy: {panel_accuracy_pct:.1f}%')
    print('Saved:', summary_path)
    return summary

## 📈 Section 6 — Evaluate the baseline on the held-out test split (graded once)

The 101-image test split is used here for the first and only time for this checkpoint,
after training has finished, under the final frozen-backbone weights — no earlier cell
in this notebook has touched it.

In [10]:
baseline_eval_dir = os.path.join(BASELINE_DIR, 'test_eval')
BASELINE_SUMMARY = evaluate_checkpoint(BASELINE_BEST_PT, 'baseline', baseline_eval_dir)

# Copy the confusion-matrix CSVs/PNGs and the summary into the flat BASELINE_DIR layout
# the brief asks for (weights, results.csv, confusion-matrix images, config.yaml,
# metrics_summary.json all directly under experiments/baseline/).
for key in ('confusion_matrix_raw_csv', 'confusion_matrix_normalized_csv',
            'confusion_matrix_raw_png', 'confusion_matrix_normalized_png'):
    src = BASELINE_SUMMARY.get(key)
    if src and os.path.exists(src):
        shutil.copy2(src, os.path.join(BASELINE_DIR, 'confusion_matrix', os.path.basename(src)))

shutil.copy2(BASELINE_SUMMARY['summary_path'], os.path.join(BASELINE_DIR, 'metrics_summary.json'))
print('Saved:', os.path.join(BASELINE_DIR, 'metrics_summary.json'))
print('Panel accuracy formula:', BASELINE_SUMMARY['panel_accuracy_formula'])

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,785,792 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 78.4±174.0 MB/s, size: 566.9 KB)
val: Scanning /content/drive/MyDrive/carparts/segmentation/datasets/v3_hitl/test/labels... 101 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 101/101 4.6it/s 22.1s
val: New cache created: /content/drive/MyDrive/carparts/segmentation/datasets/v3_hitl/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.6s
                   all        101       1610      0.856      0.815      0.828      0.607      0.867      0.804       0.82      0.555
                  door         96        172      0.957      0.959      0.963      0.825      0.956      0.953      0.953      0.651
          front_bumper  

## 🔁 Section 7 — Recompute exp04's "final" numbers under the same session (read-only)

exp04's own saved artifacts (`hitl_test_per_class_metrics.csv`) only recorded per-class
seg precision/recall/mAP50 — no box metrics, no mAP50-95, and the panel accuracy printed
in that notebook's output was never saved to a machine-readable file. Rather than
leaving `baseline_vs_final.csv` partly blank, this cell loads exp04's **already-trained**
`best.pt` **read-only** and runs the identical `evaluate_checkpoint()` used for the
baseline above, on the identical test split, in this same Ultralytics session/version.

This does not retrain, modify, or overwrite anything under
`experiments/exp04_hitl_panels/` — it is a deterministic forward pass over frozen,
already-finalized weights, written only under `experiments/baseline/final_recompute_eval/`.
No leakage: exp04's weights were already finalized before this notebook ever ran; this is
not a new training/tuning event, just a reproducible re-read of the same checkpoint's test
performance under the same code used for the baseline.

In [11]:
final_recompute_dir = os.path.join(BASELINE_DIR, 'final_recompute_eval')
FINAL_SUMMARY = evaluate_checkpoint(exp04_best_pt_path, 'final_recomputed', final_recompute_dir)
print("\nNote: these numbers are recomputed under this session for a fair, same-version "
      "comparison. exp04's own originally-saved numbers remain untouched at", EXP04_DIR)

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,785,792 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 530.9±144.9 MB/s, size: 818.0 KB)
val: Scanning /content/drive/MyDrive/carparts/segmentation/datasets/v3_hitl/test/labels.cache... 101 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 101/101 32.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.7it/s 4.2s
                   all        101       1610      0.884      0.839      0.845      0.619      0.898      0.832      0.846      0.581
                  door         96        172      0.974      0.965      0.964       0.83      0.976      0.959      0.964      0.688
          front_bumper         71         74      0.924      0.827      0.851       0.65      0.969      0.838      0.86

## 📋 Section 8 — baseline_vs_final.csv

Every metric from Sections 6/7, side by side: overall box/mask precision, recall, mAP50,
mAP50-95; per-class AP50 and AP50-95 (box + mask) for all 16 classes; and panel accuracy.

In [12]:
BASELINE_LABEL = 'No-fine-tuning (frozen backbone) baseline'
FINAL_LABEL = 'Full fine-tuning (final, exp04)'

rows = []
for scope in ('box', 'mask'):
    for metric in ('precision', 'recall', 'map50', 'map50_95'):
        rows.append({
            'metric': f'{scope}_{metric}',
            BASELINE_LABEL: BASELINE_SUMMARY['overall'][scope][metric],
            FINAL_LABEL: FINAL_SUMMARY['overall'][scope][metric],
        })
rows.append({
    'metric': 'panel_accuracy_pct',
    BASELINE_LABEL: BASELINE_SUMMARY['panel_accuracy_pct'],
    FINAL_LABEL: FINAL_SUMMARY['panel_accuracy_pct'],
})

baseline_pc = {r['class']: r for r in BASELINE_SUMMARY['per_class']}
final_pc = {r['class']: r for r in FINAL_SUMMARY['per_class']}
for cls_name in CANONICAL_CLASSES:
    for metric in ('box_ap50', 'box_ap50_95', 'mask_ap50', 'mask_ap50_95'):
        rows.append({
            'metric': f'{cls_name}_{metric}',
            BASELINE_LABEL: baseline_pc.get(cls_name, {}).get(metric),
            FINAL_LABEL: final_pc.get(cls_name, {}).get(metric),
        })

comparison_df = pd.DataFrame(rows)
comparison_csv = os.path.join(BASELINE_DIR, 'baseline_vs_final.csv')
comparison_df.to_csv(comparison_csv, index=False)
print('Saved:', comparison_csv)
print(comparison_df.head(12).to_string(index=False))

Saved: /content/drive/MyDrive/carparts/segmentation/experiments/baseline/baseline_vs_final.csv
            metric  No-fine-tuning (frozen backbone) baseline  Full fine-tuning (final, exp04)
     box_precision                                   0.855617                         0.884179
        box_recall                                   0.814711                         0.838595
         box_map50                                   0.828125                         0.844880
      box_map50_95                                   0.606529                         0.618848
    mask_precision                                   0.867235                         0.898059
       mask_recall                                   0.804298                         0.832392
        mask_map50                                   0.819985                         0.846403
     mask_map50_95                                   0.554947                         0.580979
panel_accuracy_pct                                

## 🧾 Section 9 — config.yaml (every hyperparameter, freeze setting, environment)

In [13]:
import platform, torch as _torch, ultralytics as _ul

try:
    _gpu_name = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else 'cpu (no GPU detected)'
except Exception:
    _gpu_name = 'unknown'

baseline_config = {
    'experiment': BASELINE_ID,
    'ablation_of': EXP04_ID,
    'dataset_version': DATASET_VERSION,
    'reused_data_yaml': DATA_YAML_PATH,
    'split_counts': actual_counts,
    'model_source': MODEL_SOURCE,
    'imgsz': IMG_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'weight_decay': WEIGHT_DECAY,
    'batch': BATCH,
    'seed': SEED,
    'augment': AUGMENT,
    'canonical_classes': CANONICAL_CLASSES,
    'panel_classes': sorted(PANEL_CLASSES),
    'conf_thresh': CONF_THRESH,
    'iou_thresh': IOU_THRESH,
    'freeze_layers': FREEZE_LAYERS,
    'frozen_module_count': len(FROZEN_MODULE_NAMES),
    'frozen_batchnorm_count': len(FROZEN_BN_MODULE_NAMES),
    'frozen_module_names': FROZEN_MODULE_NAMES,
    'only_difference_from_exp04': (
        'freeze=10 (backbone) + a callback forcing frozen BatchNorm layers into eval() '
        'every epoch; everything else (data, split, seed, epochs, imgsz, batch, patience, '
        'weight_decay, augmentation) is identical to exp04_hitl_panels/config.yaml'
    ),
    'environment': {
        'ultralytics_version': _ul.__version__,
        'torch_version': _torch.__version__,
        'python_version': platform.python_version(),
        'gpu': _gpu_name,
    },
}
config_path = os.path.join(BASELINE_DIR, 'config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(baseline_config, f, sort_keys=False)
print('Saved:', config_path)

Saved: /content/drive/MyDrive/carparts/segmentation/experiments/baseline/config.yaml


## ✅ Summary

- Ablation of **exp04_hitl_panels**: identical data/split/recipe, backbone frozen
  (`freeze=10`, confirmed against the live model's layer list in Section 3).
- Weights: `BASELINE_DIR/weights/best.pt`, `last.pt`.
- Test-split (101 images) metrics: `BASELINE_DIR/metrics_summary.json`,
  `BASELINE_DIR/confusion_matrix/`.
- Final (exp04) numbers recomputed under this same session for a fair comparison:
  `BASELINE_DIR/final_recompute_eval/`.
- Comparison table: `BASELINE_DIR/baseline_vs_final.csv`.
- Full recipe + environment: `BASELINE_DIR/config.yaml`.
- `exp04_hitl_panels/` was never written to — only its `config.yaml`, `split_info.yaml`,
  `data.yaml`, and `weights/best.pt` were read.